Jupyter Notebook used for scraping all houses from secondary market available on olx.pl, attempt at extracting data from descriptions using LLM call, and initial preprocessing of the data

Scraping was done using requests and BeautifulSoup4, Gemini was used as NLP via google's package genai.

In [1]:
import requests
import bs4
import pandas as pd
import numpy as np
import json
import re
from google import genai
import ast
import warnings
import logging

In [43]:
i=1
page_htmls=[]
offer_urls=[]
prev_title=None
voivodeships=["swietokrzyskie","dolnoslaskie","kujawsko-pomorskie","lubelskie","lubuskie","lodzkie","malopolskie","mazowieckie","opolskie","podkarpackie","podlaskie","pomorskie","slaskie","warminsko-mazurskie","wielkopolskie","zachodniopomorskie"]
for voivodeship in voivodeships:
    i=1
    while True:
        source=requests.get(f"https://www.olx.pl/nieruchomosci/domy/sprzedaz/{voivodeship}/?page={i}&search%5Bfilter_enum_market%5D%5B0%5D=secondary").text
        soup=bs4.BeautifulSoup(source,"lxml")
        title=soup.find("title").text
        if prev_title==title:
            break
        page_htmls.append(soup)
        prev_title=title
        i+=1
    

In [45]:
links=[]
for page in page_htmls:
    ## extracting the js element where all the links are stored
    page_js=page.find( 
    "script",
    attrs={
        "id": "olx-init-config",
        "type": "text/javascript"
    }
)
    js_text=page_js.text.partition('"')[2].replace("\\","")
    js_with_links=js_text.partition('window.__PRERENDERED_STATE__= "')[2]
    js_with_links=js_with_links.partition('";')[0]
    js_with_links=js_with_links.replace("u002F","/") # changing unicode slash to actual slash for my regex to work
    links_page=re.findall(r"https://www\.olx\.pl[^\s\"'>]*", js_with_links)
    links_page= [link for link in links_page if 'https://www.olx.pl/d' in link] # remove trash URLs
    links.extend(links_page)

In [50]:
descriptionList=[]
priceList=[]
currencyList=[]
latList=[]
longList=[]
cityList=[]
regionList=[]
paramsDictList=[]




for offer in links:
    offerHTML=requests.get(offer).text
    offer_soup=bs4.BeautifulSoup(offerHTML,"lxml")
    offer_js=offer_soup.find(
    "script",attrs={
        "id": "olx-init-config",
        "type": "text/javascript"
    }
)
    if offer_js is None:
        links.remove(offer)
        continue
    offer_js=offer_js.text.partition('"')[2].replace("\\","")
    offer_js=offer_js.partition('window.__PRERENDERED_STATE__= "')[2]
    offer_js=offer_js.partition('";')[0]
    offer_js[:-2]
    try:
        offer_js=json.loads(offer_js)
    except:
        links.remove(offer)
        continue
    actual_offer = offer_js.get("ad", {}).get("ad", {})
    offer_description = actual_offer.get("description")
    descriptionList.append(offer_description)
    price = actual_offer.get("price", {}).get("regularPrice", {})
    offer_price = price.get("value")
    priceList.append(offer_price)
    offer_currency = price.get("currencyCode")
    currencyList.append(offer_currency)
    map_data = actual_offer.get("map", {})
    offer_coords = (
        map_data.get("lat"),
        map_data.get("lon"),
)
    latList.append(offer_coords[0])
    longList.append(offer_coords[1])
    location = actual_offer.get("location", {})
    offer_location = (location.get("cityName"),location.get("regionName"))
    cityList.append(offer_location[0])
    regionList.append(offer_location[1])
    params = actual_offer.get("params", [])
    paramsDict = {param.get("key"): param.get("normalizedValue") for param in params}
    paramsDictList.append(paramsDict)

In [ ]:
for offer in links:
    offerHTML=requests.get(offer).text
    offer_soup=bs4.BeautifulSoup(offerHTML,"lxml")
    offer_js=offer_soup.find(
    "script",attrs={
        "id": "olx-init-config",
        "type": "text/javascript"
    }
)
    offer_js=offer_js.text.partition('"')[2].replace("\\","")
    offer_js=offer_js.partition('window.__PRERENDERED_STATE__= "')[2]
    offer_js=offer_js.partition('";')[0]
    offer_js[:-2]
    try:
        offer_js=json.loads(offer_js)
    except:
        links.remove(offer)
        continue

In [52]:
df=pd.DataFrame({"Description" : descriptionList, "Voivodeship" : regionList, "City" : cityList, "Latitude" : latList, "Longitude" : longList, "Price": priceList, "Currency" : currencyList, "params" : paramsDictList })
df=df.join(pd.json_normalize(df["params"]))

In [12]:
LLMClient=genai.Client(api_key="REDACTED") ## Key needs to be updated to use this script

In [9]:
class SuppressThoughtSignatureWarning(logging.Filter):
    def filter(self, record):
        # Suppress the specific warning about non-text parts
        return "there are non-text parts in the response: ['thought_signature']" not in record.getMessage()

In [10]:
desclist=df["Description"].tolist()
for desc in desclist:
    if isinstance(desc,str):
        desc.replace("\x00", "")
warnings.filterwarnings(action='ignore')
warnings.simplefilter('ignore')
logging.getLogger("google_genai.types").addFilter(SuppressThoughtSignatureWarning())
## tried various remedies for reoccuring warning about thought-signature, creating a custom filter class worked

Had some problems when trying to extract data from descriptions by sending a batch, decided to send them one by one, but it takes a lot of time, maybe there is a way to make it robust using batch or somehow run it in parallel.

In [13]:
DescDictList=[]
for desc in desclist:
    LLMDescOutput=LLMClient.models.generate_content(model="gemini-3-flash-preview",
                                  config=genai.types.GenerateContentConfig(
                                      systemInstruction="Jesteś osobistym asystentem osoby szukającej domu do kupienia. Twoim zadaniem jest analiza opisu ogłoszenia i wyciągnięcie z niego informacji. W tym celu zostanie ci przekazana lista opisów ogłoszeń domów na sprzedaż. Wyciągnij z podanego tekstu następujące informacje: przeznaczenie nieruchomości, liczba pokoi, liczba sypialni, liczba łazienek, czy posiada garaż, powierzchnia garażu, czy posiada piwnicę, powierzchnia piwnicy, rok budowy nieruchomości, rok ostatniego remontu nieruchomości,  materiał z którego dom został zbudowany, klasa energetyczna,stan domu,sposób ogrzewania. Przeanalizuj opis ogłoszenia i oceń przeznaczenie nieruchomości (Estate type)  nadając jedną z poniższych kategorii - (dom mieszkalny, pomieszczenie gospodarcze,obiekt przemysłowy,kamienica/blok,garaż). Liczba pokoi - Number of rooms powinna być równa liczbie wszystkich pomieszczeń umieszczonych w ogłoszeniu, które nie są łazienką, sypialnią bądź garażem. Obecność garażu lub piwnicy musi zostać odnotowana wartością 1, ich brak wartością 0. Oceń stan domu przypisując jedną z poniższych kategorii: (surowy,do remontu generalnego, do zamieszkania, luksusowy).  Odpowiedź powinna zostać zwrócona w formie listy obiektów typu dict w języku Python. Każdy z obiektów dict powinien mieć strukturę {\"Estate type\":\"\",\"Number of rooms\":\"\",\"Number of bedrooms\":\"\",\"Number of bathrooms\":\"\",\"Has a Garage\":\"\",\"Garage area\":\"\",\"Has a Basement\":\"\",\"Basement area\":\"\",\"Year Built\":\"\",\"Year Renovated\":\"\",\"Material\":\"\",\"Energy Class\":,\"Condition\":,\"Heating system\":} \") Jeżeli w opisie nie występują informacje, których można użyć do wypełnienia polaw obiekcie typu dict użyj wartości None"),
                                  contents=desc)
    DescDictList.append(LLMDescOutput.text)

ReadError: [WinError 10053] Nawiązane połączenie zostało przerwane przez oprogramowanie zainstalowane w komputerze-hoście

In [55]:
CleanDescDicts=[]
wrong_desc_index=[]
for i,desc in enumerate(DescDictList):
    try:
        CleanDescDicts.append(ast.literal_eval(desc[desc.find("{"):desc.find("}")+1]))
    except:
        CleanDescDicts.append(json.loads(desc[desc.find("{"):desc.find("}")+1]))
## have to try two different parsing methods, as some objects are python dictionaries, others are JSON objects

In [59]:
df=df.join(pd.json_normalize(CleanDescDicts))

In [21]:
df=df.drop(columns=["developer_url","market","Description","params","Energy Class","Currency"]) #dropped useless, mostly null, already transformed and zero-variance predictors

In [61]:
floor_mappping = {"floor_0" : "1-storey","floor_0plus" : "1.5-storey", "floor_1": "2-storey", "floor_2": "3 or more stories"}
type_mappping = {"wolnostojacy" : "detached","blizniak" : "semi-detached", "szeregowiec": "terraced", "gospodarstwo": "household", "letniskowy": "summerhouse", "pozostale": "other"}
#for material i need to prepare a regex condition, that will search for certain phrases
regex_list=[df["Material"].str.contains(pat=r"(miesz| i |,| oraz)",regex=True,case=False,na=False).to_numpy(), # matches combination of different material - "miesz"- beginning of polish word mixed, i- polish for and, "," , "oraz" - also a synonim for and
            df["Material"].str.contains(pat=r"drew",regex=True,case=False,na=False).to_numpy(), #matches anything related to wood, any word in Polish that describes wooden elements include phrase "drew"
            df["Material"].str.contains(pat=r"pustak|beton|blo(c|k)|Porotherm|Suporex|Siporex|h+h|solbet",regex=True,case=False,na=False).to_numpy(), #matches different types of concrete(beton), breezeblocks(pustak, blok, bloczek) etc. 
            df["Material"].str.contains(pat=r"murowa|tradyc",regex=True,case=False,na=False).to_numpy(),# matches unspecified types of buildings similar to previous category
            df["Material"].str.contains(pat=r"ceg(i|l|ł).*",regex=True,case=False,na=False).to_numpy() # matches brick buildings
]
material_processed_list=["Mixed","Wood","Concrete/Breezeblocks","Concrete (unspecified)","Brick"]
df["Material"]=np.where(df["Material"].isna(),None,np.select(regex_list,material_processed_list,default="Other"))
df["floor_select"]=df["floor_select"].map(floor_mappping)
df["builttype"]=df["builttype"].map(type_mappping)

In [62]:
df.to_csv(r"F:\projects\bigOLXDataset.csv")